# ContractLens: evaluation run on Kaggle / Colab

Settings: **GPU T4** and **Internet on**. Runs Gemma locally through Ollama, so no paid API is used. Total time is about 40 to 60 minutes.

In [ ]:
REPO_URL = "https://github.com/altair098/contractlens.git"
!git clone -q $REPO_URL
%cd contractlens
!pip install -q -e .

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time
os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ.copy())
time.sleep(5)
!ollama pull gemma3:4b
!ollama pull gemma3:12b

In [ ]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
!python scripts/prepare_data.py --n-contracts 10
!python scripts/build_index.py

In [ ]:
import json
first_id = json.loads(open("data/contracts.jsonl").readline())["doc_id"]
!python scripts/ask.py "Which state's law governs this agreement?" --contract-id {first_id}

## Experiments

Each run is logged to MLflow with its parameters, metrics, prompt and per-question results.

In [ ]:
!python scripts/run_eval.py --run-name gemma4b-v1-baseline --prompt-version v1 --no-verifier
!python scripts/run_eval.py --run-name gemma4b-v2-prompt --prompt-version v2 --no-verifier
!python scripts/run_eval.py --run-name gemma4b-v2-verifier --prompt-version v2 --trace
!python scripts/run_eval.py --run-name gemma4b-v2-verifier-escalate12b --prompt-version v2 --escalation-model ollama_chat/gemma3:12b

In [ ]:
!python scripts/threshold_sweep.py results/gemma4b-v2-prompt.csv

In [ ]:
!python scripts/compare_runs.py

In [ ]:
from IPython.display import Image
Image("results/gemma4b-v2-prompt_threshold_sweep.png")

In [ ]:
!zip -qr contractlens_results.zip results mlflow.db mlruns
print("Download contractlens_results.zip from the output panel")